# Risk Stratified Table

**This notebook generates a Table 1 for patients with metastatic clear cell RCC receiving first-line combination immunotherapy or single-agent antiangenic therpay, showing covariate differences stratified by the crossover point.** 

In [1]:
import numpy as np
import pandas as pd

from tableone import TableOne

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
4208,FF6F140C368B0,tki,2017-01-09
4609,F83B2011074A0,tki,2014-02-01
546,FC16EE29F46E1,ioio,2019-12-03


In [4]:
treatment_df.shape

(4831, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'ioio').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/ioio_tki_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/ioio_tki_features.csv', dtype = dtype_map)

In [7]:
features_df.shape

(3425, 162)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.head(3)

,PatientID,psurv_180_calibrated
0,FC9B9EAE7E830,0.955294
1,F21F69F4FAC6C,0.953112
2,FA6D690267BA2,0.977397


In [10]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [11]:
df.shape

(3425, 165)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(3425, 166)

In [14]:
df.neutrophil

0       7.176
1         NaN
2         NaN
3       3.400
4       5.500
        ...  
3420      NaN
3421      NaN
3422    2.400
3423    6.700
3424      NaN
Name: neutrophil, Length: 3425, dtype: float64

## Binning relevant covariates

In [15]:
df['weight_loss_5p'] = np.where(df['percent_change_weight'] <= -5, 1, 0)

In [16]:
df['creatinine_2'] = np.where(df['creatinine'] > 2, 1, 0)

In [17]:
df['hemoglobin_12'] = np.where(df['hemoglobin'] < 12, 1, 0)

In [18]:
df['platelet_150'] = np.where(df['platelet'] > 400, 1, 0)

In [19]:
df['calcium_10'] = np.where(df['calcium'] > 10, 1, 0)

In [20]:
df['neutrophil_7'] = np.where(df['neutrophil'] > 7, 1, 0)

In [21]:
df['ast_60'] = np.where(df['ast'] > 60, 1, 0)

In [22]:
df['albumin_3'] = np.where(df['albumin'] < 30, 1, 0)

In [23]:
df['alp_200'] = np.where(df['alp'] > 200, 1, 0)

In [24]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate = float(f.read())

In [25]:
df['above_threshold'] = np.where(df['psurv_180_calibrated'] >= crossover_survival_estimate, 1, 0)

## Creating table 

In [26]:
table1 = TableOne(
    data = df, 
    columns = ['age', 'GroupStage', 'ecog_index', 'days_to_treatment_before_1y', 'weight_loss_5p', 'creatinine_2', 'hemoglobin_12', 'platelet_150', 'calcium_10',  'ast_60', 'neutrophil_7', 'albumin_3', 'alp_200', 'liver_met', 'gi_met_combined', 'thoracic_met', 'brain_met', 'bone_met'],
    categorical = ['GroupStage', 'ecog_index', 'days_to_treatment_before_1y', 'weight_loss_5p', 'creatinine_2', 'hemoglobin_12', 'platelet_150', 'calcium_10',  'ast_60', 'neutrophil_7', 'albumin_3', 'alp_200', 'liver_met', 'gi_met_combined', 'thoracic_met', 'brain_met', 'bone_met'],
    continuous = ['age'],
    nonnormal = ['age'],
    groupby = 'above_threshold',
    pval = True)

In [27]:
table1

Grouped by above_threshold                                                              
                                                              Missing           Overall                 0                 1 P-Value
n                                                                                  3425               604              2821        
age, median [Q1,Q3]                                                 0  66.0 [59.0,73.0]  69.0 [61.0,76.0]  66.0 [59.0,73.0]  <0.001
GroupStage, n (%)                  I - III                                  1695 (49.5)        132 (21.9)       1563 (55.4)  <0.001
                                   IV                                       1641 (47.9)        454 (75.2)       1187 (42.1)        
                                   Unknown                                     89 (2.6)          18 (3.0)          71 (2.5)        
ecog_index, n (%)                  0.0                                      2375 (69.3)        288 (47.7)       2087 (74.0)  <0.001
                                   1.0                                       727 (21.2)        137 (22.7)        590 (20.9)        
                                   2.0                                        250 (7.3)        126 (20.9)         124 (4.4)        
                                   3.0                                         67 (2.0)          49 (8.1)          18 (0.6)        
                                   4.0                                          6 (0.2)           4 (0.7)           2 (0.1)        
days_to_treatment_before_1y, n (%) 0                                        1519 (44.4)        117 (19.4)       1402 (49.7)  <0.001
                                   1                                        1906 (55.6)        487 (80.6)       1419 (50.3)        
weight_loss_5p, n (%)              0                                        3050 (89.1)        448 (74.2)       2602 (92.2)  <0.001
                                   1                                         375 (10.9)        156 (25.8)         219 (7.8)        
creatinine_2, n (%)                0                                        3239 (94.6)        541 (89.6)       2698 (95.6)  <0.001
                                   1                                          186 (5.4)         63 (10.4)         123 (4.4)        
hemoglobin_12, n (%)               0                                        2394 (69.9)        175 (29.0)       2219 (78.7)  <0.001
                                   1                                        1031 (30.1)        429 (71.0)        602 (21.3)        
platelet_150, n (%)                0                                        3072 (89.7)        428 (70.9)       2644 (93.7)  <0.001
                                   1                                         353 (10.3)        176 (29.1)         177 (6.3)        
calcium_10, n (%)                  0                                        3084 (90.0)        506 (83.8)       2578 (91.4)  <0.001
                                   1                                         341 (10.0)         98 (16.2)         243 (8.6)        
ast_60, n (%)                      0                                        3362 (98.2)        567 (93.9)       2795 (99.1)  <0.001
                                   1                                           63 (1.8)          37 (6.1)          26 (0.9)        
neutrophil_7, n (%)                0                                        3083 (90.0)        428 (70.9)       2655 (94.1)  <0.001
                                   1                                         342 (10.0)        176 (29.1)         166 (5.9)        
albumin_3, n (%)                   0                                        3285 (95.9)        485 (80.3)       2800 (99.3)  <0.001
                                   1                                          140 (4.1)        119 (19.7)          21 (0.7)        
alp_200, n (%)                     0                                        3296 (9

In [28]:
table1.to_csv('../outputs/risk_stratified_table1.csv', header = True)

In [29]:
df.groupby('above_threshold')['ecog_index_na'].sum()

above_threshold
0     211
1    1360
Name: ecog_index_na, dtype: int64